# Task 1 - Data Merge and EDA

Merging `customer_social_profiles` and `customer_transactions` into one dataset, then
exploring it before building the product recommendation model.

Samuel Kwizera Ihimbazwe

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, chi2_contingency

pd.set_option('display.width', 150)
sns.set_style('whitegrid')

## 1. The two raw tables

The join is not obvious, so it is worth looking at both keys before merging anything.

In [ ]:
social = pd.read_csv('../data/raw/customer_social_profiles.csv')
trans = pd.read_csv('../data/raw/customer_transactions.csv')

print('social', social.shape)
display(social.head())
print('transactions', trans.shape)
display(trans.head())

In [ ]:
print('social dtypes'); print(social.dtypes)
print()
print('transaction dtypes'); print(trans.dtypes)
print()
print('social duplicate rows:', social.duplicated().sum())
print('transaction duplicate rows:', trans.duplicated().sum())
print()
print('nulls'); print(trans.isna().sum()[trans.isna().sum() > 0])

### The join key problem

The two tables do not share a key. Social profiles use `customer_id_new` (`A178`, a string)
and transactions use `customer_id_legacy` (`178`, an integer). Stripping the `A` puts both
onto the same `100-199` range, and there is no mapping table or any other candidate key in
either file, so that is the only join available.

It is worth being honest about how strong that assumption is.

In [ ]:
sid = social['customer_id_new'].str.extract(r'(\d+)')[0].astype(int)
tid = trans['customer_id_legacy']

a, b = set(sid), set(tid)
overlap = len(a & b)
expected = len(a) * len(b) / 100  # ids span 100-199, so 100 possible values

print(f'social unique ids:       {len(a)}')
print(f'transaction unique ids:  {len(b)}')
print(f'overlap after stripping A: {overlap}')
print(f'overlap expected if the two id sets were unrelated: {expected:.1f}')

The observed overlap is almost exactly what two unrelated id sets drawn from the same
range would give. So the id values themselves are no evidence that `A178` and `178` are
the same person. I am still joining on it because it is the only key the data offers, but
it is an assumption the rest of the analysis inherits, not a verified fact.

## 2. Merging

Both tables have several rows per customer, so a direct join multiplies transactions
against profiles and invents rows that never happened. Aggregating social to one row per
customer first keeps the output at one row per real transaction.

In [ ]:
naive = trans.merge(social.assign(cid=sid), left_on='customer_id_legacy', right_on='cid')
print('what a direct join would give:')
print(f'  {len(trans)} transactions -> {len(naive)} rows')
print(f'  duplicated transaction_ids: {naive["transaction_id"].duplicated().sum()}')

In [ ]:
from src.preprocessing import build

df = build()

Row count holds at 150 and every `transaction_id` is still unique, so the merge added
social context without fabricating transactions. A left join keeps the 33 transactions
whose customer has no social profile; `has_social_profile` marks them so the model can
tell a filled value apart from a real one.

## 3. Summary statistics

In [ ]:
print(df.dtypes)
print()
display(df.describe())
display(df.describe(include='object'))

## 4. Plot 1 - distributions

What the target and the main numeric drivers look like.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

order = df['product_category'].value_counts().index
sns.countplot(data=df, y='product_category', order=order, ax=axes[0], color='steelblue')
axes[0].set_title('Product category (target)')
axes[0].set_xlabel('transactions')
axes[0].set_ylabel('')

sns.histplot(df['purchase_amount'], bins=20, ax=axes[1], color='steelblue')
axes[1].set_title('Purchase amount')
axes[1].set_xlabel('amount')

sns.histplot(df['engagement_score'], bins=20, ax=axes[2], color='steelblue')
axes[2].set_title('Engagement score (customer mean)')
axes[2].set_xlabel('score')

plt.tight_layout()
plt.show()

print(df['product_category'].value_counts())

The target is close to balanced across five categories, so accuracy is a fair metric here
and no class weighting is needed. The largest class is 23.3% of the data, which is the
baseline any model has to beat to be worth anything.

`purchase_amount` is broadly flat rather than bell shaped, and engagement score piles up
in the middle because unmatched customers were filled with the median.

## 5. Plot 2 - outliers

In [ ]:
num = ['purchase_amount', 'customer_rating', 'engagement_score',
       'purchase_interest_score', 'amount_vs_customer_avg']

fig, axes = plt.subplots(1, len(num), figsize=(16, 4))
for ax, col in zip(axes, num):
    sns.boxplot(y=df[col], ax=ax, color='steelblue')
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

for col in num:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f'{col:26s} {n} outliers by 1.5*IQR')

The only column with real outliers is `amount_vs_customer_avg`, and those are customers
with a single transaction or a very uneven spend, not data errors. Nothing here needs
clipping or removal.

## 6. Plot 3 - correlations

In [ ]:
corr = df.select_dtypes('number').drop(columns=['transaction_id', 'customer_id']).corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'label': 'Pearson r'})
plt.title('Correlation between numeric features')
plt.tight_layout()
plt.show()

Nothing correlates with anything except where I built the relationship myself
(`purchase_amount` with `amount_vs_customer_avg`, and the engagement/interest pair coming
from the same aggregation). The social features and the transaction features are
essentially independent of each other.

## 7. Plot 4 - do any features actually separate the target?

The correlation heatmap only covers numeric pairs, and the target is categorical, so it
cannot show whether the features predict the product. Testing that directly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['purchase_amount', 'engagement_score', 'purchase_interest_score']):
    sns.boxplot(data=df, x='product_category', y=col, ax=ax, color='steelblue')
    ax.set_title(f'{col} by product category', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print('ANOVA - numeric feature vs product_category')
for col in ['purchase_amount', 'customer_rating', 'engagement_score',
            'purchase_interest_score', 'customer_avg_amount', 'amount_vs_customer_avg']:
    groups = [g[col].values for _, g in df.groupby('product_category')]
    f, p = f_oneway(*groups)
    print(f'  {col:26s} F={f:6.3f}  p={p:.3f}')

print()
print('chi-square - categorical feature vs product_category')
for col in ['social_media_platform', 'review_sentiment', 'has_social_profile']:
    chi2, p, _, _ = chi2_contingency(pd.crosstab(df[col], df['product_category']))
    print(f'  {col:26s} chi2={chi2:6.3f}  p={p:.3f}')

## What the EDA says

Every feature comes back non-significant against the target. The boxplots line up at the
same level across all five categories, no numeric feature separates them (all p > 0.05),
and neither platform nor sentiment shifts the mix.

So `product_category` in this dataset is close to independent of everything available to
predict it. That is a property of the data, not of the merge or the feature engineering,
and no extra features would change it.

This sets the expectation for the model: the majority class is 23.3% and random guessing
over five categories is 20%, so a fairly evaluated model should land around there. Any
score much above that would mean the target had leaked into the features or that the
model was being scored on data it trained on. I would rather report a model that scores
near baseline for a reason I can explain than one that looks good and is wrong.